##  Imports

In [ ]:
!pip install pyPDF2

In [ ]:
import os
import math
import string
from collections import Counter
from PyPDF2 import PdfReader

## PDF Text Extraction

In [ ]:
def extract_text_from_pdf(pdf_path):
    """Extract raw text from a PDF file."""
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + " "
    return text

---
## STEP 1 — Preprocessing

The preprocessing pipeline consists of three stages:
1. **Punctuation Removal**  strip all punctuation characters
2. **Tokenization + Case Normalization**  lowercase and split on whitespace
3. **Stopword Removal**  remove common, uninformative words

In [ ]:
def tokenize(text) -> list:
    """Tokenize: lowercase + split on whitespace."""
    return text.strip().lower().split()


def remove_punctuation(text) -> str:
    """Remove all punctuation characters from text."""
    return text.translate(str.maketrans('', '', string.punctuation))


STOPWORDS = set([
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves",
    "you", "your", "yours", "yourself", "yourselves", "he", "him",
    "his", "himself", "she", "her", "hers", "herself", "it", "its",
    "itself", "they", "them", "their", "theirs", "themselves", "what",
    "which", "who", "whom", "this", "that", "these", "those", "am",
    "is", "are", "was", "were", "be", "been", "being", "have", "has",
    "had", "having", "do", "does", "did", "doing", "a", "an", "the",
    "and", "but", "if", "or", "because", "as", "until", "while", "of",
    "at", "by", "for", "with", "about", "against", "between", "into",
    "through", "during", "before", "after", "above", "below", "to",
    "from", "up", "down", "in", "out", "on", "off", "over", "under",
    "again", "further", "then", "once", "here", "there", "when",
    "where", "why", "how", "all", "any", "both", "each", "few", "more",
    "most", "other", "some", "such", "no", "nor", "not", "only", "own",
    "same", "so", "than", "too", "very", "s", "t", "can", "will",
    "just", "don", "should", "now"
])


def is_stop(word) -> bool:
    """Return True if word is a stopword."""
    return word in STOPWORDS


def stopword_removal(token_list) -> list:
    """Remove stopwords from token list."""
    return [word for word in token_list if not is_stop(word)]


def preprocess(text) -> list:
    """Full preprocessing pipeline: punctuation removal → tokenize → stopword removal."""
    text   = remove_punctuation(text)          # Step 1: Remove punctuation
    tokens = tokenize(text)                    # Step 2: Lowercase + tokenize
    tokens = stopword_removal(tokens)          # Step 3: Remove stopwords
    tokens = [t for t in tokens if len(t) > 1] # Remove single-char tokens
    return tokens

---
## STEP 2  TF-IDF Equations

### Term Frequency (TF)
$$TF(t, d) = \frac{\text{count}(t \text{ in } d)}{\text{total\_terms}(d)}$$

Measures how frequently a term appears in a document, normalized by total terms.

### Inverse Document Frequency (IDF) — Smoothed
$$IDF(t, D) = \log\left(\frac{1 + N}{1 + df(t)}\right) + 1$$

Where $N$ = number of documents, $df(t)$ = number of documents containing term $t$.

### TF-IDF Score
$$TF\text{-}IDF(t, d, D) = TF(t, d) \times IDF(t, D)$$

High TF-IDF → term is frequent in *this* document but rare across the corpus.

In [ ]:
def compute_tf(token_list) -> dict:
    """
    Term Frequency (TF):
        TF(t, d) = count(t in d) / total_terms(d)
    """
    total_terms = len(token_list)
    term_counts = Counter(token_list)
    tf = {term: count / total_terms for term, count in term_counts.items()}
    return tf


def compute_idf(all_token_lists) -> dict:
    """
    Inverse Document Frequency (IDF) — smoothed variant:
        IDF(t, D) = log( (1 + N) / (1 + df(t)) ) + 1
    """
    N = len(all_token_lists)
    df = {}
    for token_list in all_token_lists:
        unique_terms = set(token_list)
        for term in unique_terms:
            df[term] = df.get(term, 0) + 1

    idf = {}
    for term, doc_freq in df.items():
        idf[term] = math.log((1 + N) / (1 + doc_freq)) + 1
    return idf


def compute_tfidf(tf, idf) -> dict:
    """
    TF-IDF Score:
        TF-IDF(t, d, D) = TF(t, d) * IDF(t, D)
    """
    tfidf = {}
    for term, tf_score in tf.items():
        idf_score = idf.get(term, 0)
        tfidf[term] = tf_score * idf_score
    return tfidf

---
## STEP 3 — Document Embedding (TF-IDF Vectors)

Each document is represented as a vector in the vocabulary space. Terms not present in a document get a score of 0.

In [ ]:
def build_tfidf_vectors(tfidf_list, vocabulary):
    """
    Build a TF-IDF matrix: each document becomes a fixed-length vector
    aligned with the shared vocabulary.

    Returns:
        matrix (list of lists): shape = [n_docs, vocab_size]
    """
    vocab_list = sorted(vocabulary)   # deterministic ordering
    matrix = []
    for tfidf in tfidf_list:
        vector = [tfidf.get(term, 0.0) for term in vocab_list]
        matrix.append(vector)
    return vocab_list, matrix

In [ ]:
print("=" * 60)
print("   TF-IDF FROM SCRATCH — Full Pipeline")
print("=" * 60)

# ── Document paths ─────────────────────────────────────────────
pdf_paths = [
    r"D:\Stuff\Study\2nd term\NLP\doc1_machine_learning.pdf",
    r"D:\Stuff\Study\2nd term\NLP\doc2_climate_change.pdf",
    r"D:\Stuff\Study\2nd term\NLP\doc3_programming_languages.pdf",
   r"D:\Stuff\Study\2nd term\NLP\doc4_human_body.pdf",
]
doc_names = [os.path.basename(p) for p in pdf_paths]

# ── Extract & Preprocess ───────────────────────────────────────
print("\n[1] Extracting and preprocessing text...")
raw_texts   = []
token_lists = []

for path, name in zip(pdf_paths, doc_names):
    raw    = extract_text_from_pdf(path)
    tokens = preprocess(raw)
    raw_texts.append(raw)
    token_lists.append(tokens)
    print(f"  {name}: {len(tokens)} tokens after preprocessing")

# ── Compute TF, IDF, TF-IDF ────────────────────────────────────
print("\n[2] Computing TF, IDF, and TF-IDF scores...")
tf_list    = [compute_tf(tokens)     for tokens in token_lists]
idf        =  compute_idf(token_lists)
tfidf_list = [compute_tfidf(tf, idf) for tf in tf_list]
print("    Done.")

   TF-IDF FROM SCRATCH — Full Pipeline

[1] Extracting and preprocessing text...
  doc1_machine_learning.pdf: 157 tokens after preprocessing
  doc2_climate_change.pdf: 149 tokens after preprocessing
  doc3_programming_languages.pdf: 144 tokens after preprocessing
  doc4_human_body.pdf: 162 tokens after preprocessing

[2] Computing TF, IDF, and TF-IDF scores...
    Done.


---
## Top TF-IDF Words per Document

In [ ]:
TOP_N = 10

print(f"Top {TOP_N} TF-IDF terms per document\n" + "=" * 60)

for name, tfidf in zip(doc_names, tfidf_list):
    top_terms = sorted(tfidf.items(), key=lambda x: x[1], reverse=True)[:TOP_N]
    print(f"\n {name}")
    print(f"  {'Term':<25} {'TF-IDF Score':>12}")
    print("  " + "-" * 38)
    for term, score in top_terms:
        print(f"  {term:<25} {score:>12.6f}")

Top 10 TF-IDF terms per document

 doc1_machine_learning.pdf
  Term                      TF-IDF Score
  --------------------------------------
  learning                      0.115477
  data                          0.067362
  algorithms                    0.048823
  machine                       0.048115
  supervised                    0.036617
  training                      0.036617
  models                        0.036617
  neural                        0.036617
  networks                      0.036617
  like                          0.025478

 doc2_climate_change.pdf
  Term                      TF-IDF Score
  --------------------------------------
  carbon                        0.077166
  global                        0.051444
  climate                       0.038583
  change                        0.038583
  dioxide                       0.038583
  energy                        0.038583
  temperatures                  0.025722
  weather                       0.025722
  since    

---
##  TF-IDF Document Embedding (Vector Representation)

In [ ]:
# Build shared vocabulary across all documents
vocabulary = set()
for token_list in token_lists:
    vocabulary.update(token_list)

vocab_list, tfidf_matrix = build_tfidf_vectors(tfidf_list, vocabulary)

print(f"Vocabulary size : {len(vocab_list)} unique terms")
print(f"Matrix shape    : {len(tfidf_matrix)} docs × {len(vocab_list)} terms")
print("\nEach row = one document's TF-IDF embedding vector.")
print("(Showing first 10 dimensions of each vector)\n")

for name, vector in zip(doc_names, tfidf_matrix):
    preview = ", ".join(f"{v:.4f}" for v in vector[:10])
    print(f"  {name}\n    [{preview}, ...]\n")

Vocabulary size : 445 unique terms
Matrix shape    : 4 docs × 445 terms

Each row = one document's TF-IDF embedding vector.
(Showing first 10 dimensions of each vector)

  doc1_machine_learning.pdf
    [0.0000, 0.0122, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, ...]

  doc2_climate_change.pdf
    [0.0000, 0.0000, 0.0000, 0.0129, 0.0129, 0.0000, 0.0129, 0.0129, 0.0129, 0.0129, ...]

  doc3_programming_languages.pdf
    [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0133, 0.0000, 0.0000, 0.0000, 0.0000, ...]

  doc4_human_body.pdf
    [0.0237, 0.0000, 0.0118, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, ...]



---
## IDF Values — Shared Vocabulary

Terms with **higher IDF** are rarer across the corpus (better discriminators).  
Terms with **lower IDF** appear in many documents (less informative).

In [ ]:
# Show the 15 highest and 15 lowest IDF terms
sorted_idf = sorted(idf.items(), key=lambda x: x[1], reverse=True)

print("Top 15 highest IDF terms (most corpus-unique):")
print(f"  {'Term':<25} {'IDF':>8}")
print("  " + "-" * 34)
for term, score in sorted_idf[:15]:
    print(f"  {term:<25} {score:>8.4f}")

print("\nTop 15 lowest IDF terms (most common across corpus):")
print(f"  {'Term':<25} {'IDF':>8}")
print("  " + "-" * 34)
for term, score in sorted_idf[-15:]:
    print(f"  {term:<25} {score:>8.4f}")

Top 15 highest IDF terms (most corpus-unique):
  Term                           IDF
  ----------------------------------
  clustering                  1.9163
  group                       1.9163
  support                     1.9163
  l1                          1.9163
  component                   1.9163
  neural                      1.9163
  regularization              1.9163
  f1score                     1.9163
  enables                     1.9163
  together                    1.9163
  algorithms                  1.9163
  image                       1.9163
  models                      1.9163
  performance                 1.9163
  common                      1.9163

Top 15 lowest IDF terms (most common across corpus):
  Term                           IDF
  ----------------------------------
  information                 1.5108
  causing                     1.5108
  dramatically                1.5108
  times                       1.5108
  human                       1.5108
  longterm 

---
##  Full TF, IDF, and TF-IDF Raw Outputs

In [ ]:
print("TF scores (term frequency per document):")
for name, tf in zip(doc_names, tf_list):
    print(f"  {name}: {len(tf)} unique terms")
print()
print(f"IDF scores: {len(idf)} total unique terms in corpus")
print()
print("TF-IDF scores per document:")
for name, tfidf in zip(doc_names, tfidf_list):
    print(f"  {name}: {len(tfidf)} scored terms")

TF scores (term frequency per document):
  doc1_machine_learning.pdf: 111 unique terms
  doc2_climate_change.pdf: 119 unique terms
  doc3_programming_languages.pdf: 110 unique terms
  doc4_human_body.pdf: 135 unique terms

IDF scores: 445 total unique terms in corpus

TF-IDF scores per document:
  doc1_machine_learning.pdf: 111 scored terms
  doc2_climate_change.pdf: 119 scored terms
  doc3_programming_languages.pdf: 110 scored terms
  doc4_human_body.pdf: 135 scored terms
